In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import joblib

In [2]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: mps


In [3]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


In [4]:
train_data = np.load(
    "../data/processed/train_sequences.npz"
)

val_data = np.load(
    "../data/processed/validation_sequences.npz"
)

test_data = np.load(
    "../data/processed/test_sequences.npz"
)

X_train = train_data["X"]
y_train = train_data["y"]

X_val = val_data["X"]
y_val = val_data["y"]

X_test = test_data["X"]
y_test = test_data["y"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (23973, 12, 207)
y_train: (23973, 207)
X_val: (3410, 12, 207)
y_val: (3410, 207)
X_test: (6838, 12, 207)
y_test: (6838, 207)


In [5]:
train_target_mask = np.load(
    "../data/processed/train_target_mask.npy"
)

val_target_mask = np.load(
    "../data/processed/validation_target_mask.npy"
)

test_target_mask = np.load(
    "../data/processed/test_target_mask.npy"
)

print("Train mask:", train_target_mask.shape)
print("Validation mask:", val_target_mask.shape)
print("Test mask:", test_target_mask.shape)

Train mask: (23973, 207)
Validation mask: (3410, 207)
Test mask: (6838, 207)


In [6]:
INPUT_STEPS = X_train.shape[1]
NUM_SENSORS = X_train.shape[2]

print("Historical time steps:", INPUT_STEPS)
print("Number of sensors:", NUM_SENSORS)

Historical time steps: 12
Number of sensors: 207


In [7]:
D_MODEL = 128
N_HEADS = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 256
DROPOUT = 0.2

print("Model dimension:", D_MODEL)
print("Attention heads:", N_HEADS)
print("Transformer layers:", NUM_LAYERS)
print("Feed-forward dimension:", DIM_FEEDFORWARD)
print("Dropout:", DROPOUT)

Model dimension: 128
Attention heads: 4
Transformer layers: 2
Feed-forward dimension: 256
Dropout: 0.2


In [8]:
class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=100,
        dropout=0.1
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        x = x + self.pe[:, :x.size(1), :]

        return self.dropout(x)

In [9]:
class TrafficTransformer(nn.Module):

    def __init__(
        self,
        num_sensors,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.2
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            num_sensors,
            d_model
        )

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=INPUT_STEPS,
            dropout=dropout
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.output_layer = nn.Linear(
            d_model,
            num_sensors
        )

    def forward(self, x):


        x = self.input_projection(x)


        x = self.positional_encoding(x)

        x = self.transformer(x)

        x = x[:, -1, :]

        x = self.output_layer(x)


        return x

In [10]:
transformer_model = TrafficTransformer(
    num_sensors=NUM_SENSORS,
    d_model=D_MODEL,
    nhead=N_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(device)

print(transformer_model)

TrafficTransformer(
  (input_projection): Linear(in_features=207, out_features=128, bias=True)
  (positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (output_layer): Linear(in_features=128, out_features=207, bias=True)
)


In [11]:
total_parameters = sum(
    parameter.numel()
    for parameter in transformer_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in transformer_model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

Total parameters: 318287
Trainable parameters: 318287


In [12]:
sample_input = torch.tensor(
    X_train[:8],
    dtype=torch.float32,
    device=device
)

sample_output = transformer_model(
    sample_input
)

print("Input shape:", sample_input.shape)
print("Output shape:", sample_output.shape)

Input shape: torch.Size([8, 12, 207])
Output shape: torch.Size([8, 207])


In [13]:
BATCH_SIZE = 64

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
    torch.tensor(train_target_mask, dtype=torch.bool)
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
    torch.tensor(val_target_mask, dtype=torch.bool)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
    torch.tensor(test_target_mask, dtype=torch.bool)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 375
Validation batches: 54
Test batches: 107


In [14]:
def masked_mse_loss(predictions, targets, mask):

    mask = mask.float()

    squared_error = (
        predictions - targets
    ) ** 2

    masked_error = (
        squared_error * mask
    )

    return (
        masked_error.sum()
        / mask.sum().clamp_min(1.0)
    )

In [15]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    transformer_model.parameters(),
    lr=0.001
)

print("Optimizer:", optimizer)

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [16]:
batch_x, batch_y, batch_mask = next(
    iter(train_loader)
)

batch_x = batch_x.to(device)
batch_y = batch_y.to(device)
batch_mask = batch_mask.to(device)

transformer_model.train()

optimizer.zero_grad()

predictions = transformer_model(
    batch_x
)

loss = masked_mse_loss(
    predictions,
    batch_y,
    batch_mask
)

loss.backward()

optimizer.step()

print("Input:", batch_x.shape)
print("Target:", batch_y.shape)
print("Prediction:", predictions.shape)
print("Loss:", loss.item())

Input: torch.Size([64, 12, 207])
Target: torch.Size([64, 207])
Prediction: torch.Size([64, 207])
Loss: 1.3206536769866943


In [17]:
torch.manual_seed(42)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(42)

transformer_model = TrafficTransformer(
    num_sensors=NUM_SENSORS,
    d_model=D_MODEL,
    nhead=N_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(device)

optimizer = torch.optim.Adam(
    transformer_model.parameters(),
    lr=0.001
)

print("Transformer reset successfully.")
print("Device:", device)

Transformer reset successfully.
Device: mps


In [18]:
NUM_EPOCHS = 20
PATIENCE = 4

best_val_loss = float("inf")
patience_counter = 0

train_history = []
val_history = []

best_model_path = "../models/transformer_best.pth"

for epoch in range(1, NUM_EPOCHS + 1):

    transformer_model.train()

    train_loss_total = 0.0
    train_batches = 0

    for batch_x, batch_y, batch_mask in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_mask = batch_mask.to(device)

        optimizer.zero_grad()

        predictions = transformer_model(batch_x)

        loss = masked_mse_loss(
            predictions,
            batch_y,
            batch_mask
        )

        loss.backward()

        optimizer.step()

        train_loss_total += loss.item()
        train_batches += 1

    train_loss = train_loss_total / train_batches

    transformer_model.eval()

    val_loss_total = 0.0
    val_batches = 0

    with torch.no_grad():

        for batch_x, batch_y, batch_mask in val_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            predictions = transformer_model(batch_x)

            loss = masked_mse_loss(
                predictions,
                batch_y,
                batch_mask
            )

            val_loss_total += loss.item()
            val_batches += 1

    val_loss = val_loss_total / val_batches

    train_history.append(train_loss)
    val_history.append(val_loss)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            transformer_model.state_dict(),
            best_model_path
        )

        print("  → Best Transformer model saved.")

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print("  → Early stopping triggered.")
            break

Epoch 01/20 | Train Loss: 0.551293 | Val Loss: 0.481801
  → Best Transformer model saved.
Epoch 02/20 | Train Loss: 0.459424 | Val Loss: 0.455263
  → Best Transformer model saved.
Epoch 03/20 | Train Loss: 0.433317 | Val Loss: 0.448315
  → Best Transformer model saved.
Epoch 04/20 | Train Loss: 0.417764 | Val Loss: 0.445568
  → Best Transformer model saved.
Epoch 05/20 | Train Loss: 0.406974 | Val Loss: 0.446023
Epoch 06/20 | Train Loss: 0.399059 | Val Loss: 0.444803
  → Best Transformer model saved.
Epoch 07/20 | Train Loss: 0.390796 | Val Loss: 0.445585
Epoch 08/20 | Train Loss: 0.385153 | Val Loss: 0.445234
Epoch 09/20 | Train Loss: 0.379107 | Val Loss: 0.445911
Epoch 10/20 | Train Loss: 0.373962 | Val Loss: 0.443041
  → Best Transformer model saved.
Epoch 11/20 | Train Loss: 0.370594 | Val Loss: 0.446773
Epoch 12/20 | Train Loss: 0.366996 | Val Loss: 0.450090
Epoch 13/20 | Train Loss: 0.363926 | Val Loss: 0.450245
Epoch 14/20 | Train Loss: 0.361007 | Val Loss: 0.451989
  → Early st

In [19]:
transformer_model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device,
        weights_only=True
    )
)

transformer_model.eval()

print("Best Transformer model restored.")
print("Best validation loss:", best_val_loss)

Best Transformer model restored.
Best validation loss: 0.4430413304103745


In [20]:
transformer_history_df = pd.DataFrame({
    "epoch": range(1, len(train_history) + 1),
    "train_loss": train_history,
    "validation_loss": val_history
})

transformer_history_df.to_csv(
    "../results/transformer_training_history.csv",
    index=False
)

print(
    "Saved: ../results/transformer_training_history.csv"
)

Saved: ../results/transformer_training_history.csv


In [21]:
transformer_model.eval()

transformer_val_predictions = []

with torch.no_grad():

    for batch_x, batch_y, batch_mask in val_loader:

        batch_x = batch_x.to(device)

        predictions = transformer_model(batch_x)

        transformer_val_predictions.append(
            predictions.cpu().numpy()
        )

transformer_val_predictions = np.concatenate(
    transformer_val_predictions,
    axis=0
)

print(
    "Transformer validation predictions:",
    transformer_val_predictions.shape
)

print(
    "Validation targets:",
    y_val.shape
)

Transformer validation predictions: (3410, 207)
Validation targets: (3410, 207)


In [25]:
import joblib

scaler = joblib.load(
    "../data/processed/traffic_scaler.pkl"
)

print("Scaler loaded successfully.")

Scaler loaded successfully.


In [26]:
transformer_val_predictions_mph = scaler.inverse_transform(
    transformer_val_predictions
)

val_targets_mph = scaler.inverse_transform(
    y_val
)

print(
    "Prediction range:",
    transformer_val_predictions_mph.min(),
    "to",
    transformer_val_predictions_mph.max(),
    "mph"
)

print(
    "Target range:",
    val_targets_mph.min(),
    "to",
    val_targets_mph.max(),
    "mph"
    
)

Prediction range: -12.901585 to 81.62326 mph
Target range: 0.4999962 to 70.0 mph


In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

valid_mask = val_target_mask.astype(bool)

transformer_val_mae = mean_absolute_error(
    val_targets_mph[valid_mask],
    transformer_val_predictions_mph[valid_mask]
)

transformer_val_rmse = np.sqrt(
    mean_squared_error(
        val_targets_mph[valid_mask],
        transformer_val_predictions_mph[valid_mask]
    )
)

print(
    "Transformer Validation MAE:",
    transformer_val_mae
)

print(
    "Transformer Validation RMSE:",
    transformer_val_rmse
)

Transformer Validation MAE: 3.4215071201324463
Transformer Validation RMSE: 6.177782230390839


In [28]:
total_predictions = transformer_val_predictions_mph.size

negative_predictions = np.sum(
    transformer_val_predictions_mph < 0
)

above_max_predictions = np.sum(
    transformer_val_predictions_mph > 70
)

print("Total predictions:", total_predictions)

print(
    "Negative predictions:",
    negative_predictions
)

print(
    "Negative prediction percentage:",
    100 * negative_predictions / total_predictions
)

print(
    "Predictions above 70 mph:",
    above_max_predictions
)

print(
    "Above-70 prediction percentage:",
    100 * above_max_predictions / total_predictions
    
    
    
)

Total predictions: 705870
Negative predictions: 433
Negative prediction percentage: 0.06134274016461955
Predictions above 70 mph: 5516
Above-70 prediction percentage: 0.7814470086559848


In [29]:
transformer_results = pd.DataFrame({
    "model": ["Transformer"],
    "validation_mae": [transformer_val_mae],
    "validation_rmse": [transformer_val_rmse]
})

transformer_results.to_csv(
    "../results/transformer_validation_results.csv",
    index=False
)

print(transformer_results)

print(
    "\nSaved to "
    "../results/transformer_validation_results.csv"
)

         model  validation_mae  validation_rmse
0  Transformer        3.421507         6.177782

Saved to ../results/transformer_validation_results.csv
